In [0]:
%run ./schemas

RAW CONTRIBUTORS SCHEMA:
struct<id:bigint,login:string,type:string,site_admin:boolean,contributions:bigint,_repo:string,_ingested_at:string,_load_type:string,_source_endpoint:string,_batch_id:string,load_timestamp:string,_corrupt_record:string>

BRONZE CONTRIBUTORS SCHEMA:
struct<record_key:string,repository:string,batch_id:string,source_file:string,source_updated_at:timestamp,raw_payload:string,payload_hash:string,load_timestamp:timestamp,ingested_at:timestamp>

SILVER CONTRIBUTORS SCHEMA:
struct<record_key:string,repository:string,contributor_id:bigint,contributions:bigint,account_type:string,snapshot_at:timestamp,batch_id:string,payload_hash:string,load_timestamp:timestamp>

All Contributors schemas are defined successfully!


In [0]:
%run ./common

In [0]:
# Cell 3: Widgets — fill these out BEFORE running the remainder.
# Remove any widgets from a previous run so new defaults take effect.
for _w in ['batch_id','run_date','load_type','manifest_path','run_scope']:
    try: dbutils.widgets.remove(_w)
    except Exception: pass
dbutils.widgets.text("batch_id", "92199a9d-c132-4525-92c9-5094e8e1f310")
dbutils.widgets.text("run_date", "2026-10-08")
dbutils.widgets.dropdown("load_type", "FULL", ["INCREMENTAL", "FULL"])
dbutils.widgets.text("manifest_path", "")


# Select which datasets the driver should process
# CORE_ONLY: Commits, Issues, and Pull Requests
# METADATA_ONLY: Repository Metadata
# CONTRIBUTORS_ONLY: Contributors
# ALL: All datasets (once integration is complete)

dbutils.widgets.dropdown(
    "run_scope",
    "ALL",
    [
        "CORE_ONLY",
        "METADATA_ONLY",
        "CONTRIBUTORS_ONLY",
        "ALL"
    ]
)

print("Parameters ready. Check the widgets at the top.")



print("Parameters ready. Check the widgets at the top of the notebook.")

Parameters ready. Check the widgets at the top.
Parameters ready. Check the widgets at the top of the notebook.


In [0]:
# Cell 4: Imports, configuration and manifest validation.
import json
import re
import time
import uuid
import traceback
from datetime import date
from pathlib import Path

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

batch_id = dbutils.widgets.get("batch_id").strip()
run_date = dbutils.widgets.get("run_date").strip()
load_type = dbutils.widgets.get("load_type").strip().upper()
manifest_override = dbutils.widgets.get("manifest_path").strip()
# Read which datasets should be processed
run_scope = dbutils.widgets.get("run_scope").strip().upper()

# Keep explicit paths; do not rely on current_date() or implicit file discovery.
if not batch_id or not re.fullmatch(r"[A-Za-z0-9_-]{1,100}", batch_id):
    raise ValueError("Set a valid batch_id using the notebook widget")
if not run_date or date.fromisoformat(run_date).isoformat() != run_date:
    raise ValueError("Set run_date as YYYY-MM-DD")
if load_type not in ("FULL", "INCREMENTAL"):
    raise ValueError("load_type must be FULL or INCREMENTAL")
# Reject unsupported processing scopes

# Validate the selected processing scope
if run_scope not in (
    "CORE_ONLY",
    "METADATA_ONLY",
    "CONTRIBUTORS_ONLY",
    "ALL"
):
    raise ValueError("Invalid run_scope widget")


manifest_dir = Path(ROOT) / "_manifests"
if manifest_override:
    manifest_file = Path(manifest_override)
elif load_type == "INCREMENTAL":
    manifest_file = manifest_dir / f"incremental_{batch_id}.json"
else:
    matches = list(manifest_dir.glob(f"full_load_*_{batch_id}.json"))
    if len(matches) != 1:
        raise ValueError("FULL manifest not found uniquely; set manifest_path widget")
    manifest_file = matches[0]

if not manifest_file.is_file():
    raise FileNotFoundError(f"Cannot find manifest: {manifest_file}")
manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
if manifest.get("batch_id") != batch_id:
    raise ValueError("Manifest batch_id does not match notebook widget")
if manifest.get("load_type") != load_type:
    raise ValueError("Manifest load_type does not match notebook widget")
if manifest.get("run_date") and manifest["run_date"] != run_date:
    raise ValueError("Manifest run_date does not match notebook widget")

expected = {(r, e) for r in REPOS for e in ENTITIES}
files = [x for x in manifest["files"] if (x.get("repository"), x.get("entity")) in expected]
seen = [(x["repository"], x["entity"]) for x in files]
if len(files) != len(expected) or set(seen) != expected or len(set(seen)) != len(seen):
    raise ValueError("Manifest must list exactly 9 distinct repo/entity JSONL files")

# Locate the Repository Metadata files when requested
metadata_files = []

if run_scope in ("METADATA_ONLY", "ALL"):

    # Metadata currently exists in our historical FULL batch
    if load_type != "FULL":
        raise ValueError(
            "Repository Metadata currently requires the FULL batch"
        )

    # Ensure schemas and metadata MERGE function are available
    if "repo_metadata" not in RAW_SCHEMAS:
        raise ValueError("Missing metadata Raw schema")

    if "repo_metadata" not in SILVER_SCHEMA:
        raise ValueError("Missing metadata Silver schema")

    if not callable(globals().get("merge_metadata_snapshot")):
        raise ValueError("Missing metadata MERGE function")

    # First check whether metadata files are listed in the manifest
    metadata_files = [
        item for item in manifest["files"]
        if item.get("entity") == "repo_metadata"
        and item.get("repository") in REPOS
    ]

    # Older manifests might not list Repository Metadata
    # Build paths from our known historical full-load naming
    if not metadata_files:
        metadata_files = [
            {
                "repository": repo,
                "entity": "repo_metadata",
                "path": str(
                    Path(ROOT)
                    / "repo_metadata"
                    / repo.replace("/", "__")
                    / f"repo_metadata_full_{run_date}_{batch_id}.jsonl"
                ),
                "records": 1
            }
            for repo in REPOS
        ]

    # Confirm exactly one file exists for each repository
    actual_repos = [item["repository"] for item in metadata_files]

    if len(metadata_files) != 3 or set(actual_repos) != set(REPOS):
        raise ValueError("Expected exactly 3 metadata files")

    # Verify file paths before processing anything
    for item in metadata_files:
        path = Path(item["path"])

        if not str(path).startswith(str(Path(ROOT)) + "/"):
            raise ValueError(f"Unexpected metadata path: {path}")

        if path.suffix != ".jsonl" or not path.is_file():
            raise FileNotFoundError(f"Missing JSONL file: {path}")

print("Repository Metadata files selected:", len(metadata_files))


# STEP 7: Locate Contributors files for the selected batch

# Keep the list empty unless Contributors processing is requested
contributors_files = []

if run_scope in ("CONTRIBUTORS_ONLY", "ALL"):

    # Our existing Contributors files belong to the historical FULL batch
    if load_type != "FULL":
        raise ValueError(
            "Contributors processing currently requires FULL load"
        )

    # Make sure the explicit Contributors schemas exist
    if "contributors" not in RAW_SCHEMAS:
        raise ValueError("Missing Contributors Raw schema")

    if "contributors" not in SILVER_SCHEMA:
        raise ValueError("Missing Contributors Silver schema")

    # Use files from the manifest when they are listed
    contributors_files = [
        item for item in manifest["files"]
        if item.get("entity") == "contributors"
        and item.get("repository") in REPOS
    ]

    # Older manifests may not list Contributors
    # Reuse the historical files already stored in the Raw Volume
    if not contributors_files:

        counts = {
            "apache/spark": 331,
            "pandas-dev/pandas": 413,
            "scikit-learn/scikit-learn": 407
        }

        contributors_files = [
            {
                "repository": repo,
                "entity": "contributors",
                "path": str(
                    Path(ROOT)
                    / "contributors"
                    / repo.replace("/", "__")
                    / f"contributors_full_{run_date}_{batch_id}.jsonl"
                ),
                "records": counts[repo]
            }
            for repo in REPOS
        ]

    # Require exactly one file per repository
    selected_repos = [
        item["repository"] for item in contributors_files
    ]

    if (
        len(contributors_files) != 3
        or set(selected_repos) != set(REPOS)
    ):
        raise ValueError("Expected exactly 3 Contributors files")

    # Verify each file exists inside our Raw Volume
    for item in contributors_files:
        path = Path(item["path"])

        if not str(path).startswith(str(Path(ROOT)) + "/"):
            raise ValueError(f"Unexpected path: {path}")

        if path.suffix != ".jsonl" or not path.is_file():
            raise FileNotFoundError(f"Missing file: {path}")

# Display selected Contributors files
print("Contributors files selected:", len(contributors_files))

for item in contributors_files:
    print(
        item["repository"],
        "->",
        item["entity"],
        "| Expected records:",
        item.get("records", "Unknown")
    )


# Deterministic order: all Bronze, then all Silver.
repo_order = {name: index for index, name in enumerate(REPOS)}
entity_order = {name: index for index, name in enumerate(ENTITIES)}
files.sort(key=lambda x: (repo_order[x["repository"]], entity_order[x["entity"]]))
root_prefix = str(Path(ROOT)) + "/"
for item in files:
    p = Path(item["path"])
    if not str(p).startswith(root_prefix) or p.suffix != ".jsonl":
        raise ValueError(f"Unexpected source path in manifest: {p}")
    if not p.is_file():
        raise FileNotFoundError(p)

print("Ready to process:", batch_id, load_type)
print("Manifest:", manifest_file)
print("Files:", len(files), "| Raw manifest records:", sum(int(x.get("records", 0)) for x in files))
for x in files:
    print("  ", x["repository"], x["entity"], x.get("records", "?"), "rows")

Repository Metadata files selected: 3
Contributors files selected: 3
apache/spark -> contributors | Expected records: 331
pandas-dev/pandas -> contributors | Expected records: 413
scikit-learn/scikit-learn -> contributors | Expected records: 407
Ready to process: 92199a9d-c132-4525-92c9-5094e8e1f310 FULL
Manifest: /Volumes/github_health/raw/bronze_data/bronze/_manifests/full_load_2026-10-08_92199a9d-c132-4525-92c9-5094e8e1f310.json
Files: 9 | Raw manifest records: 29099
   apache/spark commits 4775 rows
   apache/spark issues 110 rows
   apache/spark pulls 8403 rows
   pandas-dev/pandas commits 2777 rows
   pandas-dev/pandas issues 3135 rows
   pandas-dev/pandas pulls 4739 rows
   scikit-learn/scikit-learn commits 1054 rows
   scikit-learn/scikit-learn issues 1106 rows
   scikit-learn/scikit-learn pulls 3000 rows


In [0]:

# ============================================================
# CELL 5: BRONZE AND SILVER TRANSFORMATION FUNCTIONS
# Supports:
# 1. Commits
# 2. Issues
# 3. Pull Requests
# 4. Repository Metadata
# 5. Contributors
#
# This cell only DEFINES functions.
# It does NOT process records or write to Delta tables.
# ============================================================

import time
import uuid
import traceback

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 1. BUSINESS COLUMNS FOR CHANGE DETECTION
# ============================================================

# These fields are used to calculate payload_hash.
# Exclude batch IDs and ingestion timestamps because they
# are pipeline metadata, not business-data changes.

BUSINESS_COLS = {
    "commits": [
        "sha", "node_id", "commit",
        "author", "committer", "html_url"
    ],

    "issues": [
        "id", "number", "node_id", "title",
        "state", "created_at", "updated_at",
        "closed_at", "user", "labels", "html_url"
    ],

    "pulls": [
        "id", "number", "node_id", "title",
        "state", "draft", "created_at",
        "updated_at", "closed_at", "merged_at",
        "user", "merged", "html_url"
    ],

    "repo_metadata": [
        "id", "name", "full_name", "description",
        "html_url", "language", "default_branch",
        "stargazers_count", "forks_count",
        "open_issues_count", "subscribers_count",
        "archived", "fork", "created_at",
        "updated_at", "pushed_at", "topics", "license"
    ],

    "contributors": [
        "id", "type", "site_admin", "contributions"
    ]
}


# Repository Metadata and Contributors are snapshots.
# Their update order is based on collection time.
SNAPSHOT_ENTITIES = {"repo_metadata", "contributors"}


# ============================================================
# 2. MERGE FOR COMMITS, ISSUES AND PULL REQUESTS
# ============================================================

def merge_delta_changed(df, table_name):
    """
    Insert new records or update existing records only when
    the business payload changed and the source timestamp
    is newer than the stored timestamp.

    Prevents older historical data from overwriting newer data.
    """

    columns = df.columns

    # Values used for inserting new records
    values = {
        column: f"s.`{column}`"
        for column in columns
    }

    # Update all supplied fields except the logical primary key
    updates = {
        column: expression
        for column, expression in values.items()
        if column != "record_key"
    }

    (
        DeltaTable.forName(spark, table_name)
        .alias("t")
        .merge(
            df.alias("s"),
            "t.record_key = s.record_key"
        )
        .whenMatchedUpdate(
            condition=(
                "NOT (t.payload_hash <=> s.payload_hash) "
                "AND s.source_updated_at IS NOT NULL "
                "AND (t.source_updated_at IS NULL "
                "OR s.source_updated_at > t.source_updated_at)"
            ),
            set=updates
        )
        .whenNotMatchedInsert(values=values)
        .execute()
    )

    # Get the actual insert/update counts from Delta history
    metrics = spark.sql(
        f"DESCRIBE HISTORY {table_name} LIMIT 1"
    ).first()["operationMetrics"]

    inserted = int(metrics.get("numTargetRowsInserted", 0))
    updated = int(metrics.get("numTargetRowsUpdated", 0))

    return inserted, updated


# ============================================================
# 3. SNAPSHOT-AWARE MERGE
# ============================================================

def merge_delta_metadata_snapshot(
    df, table_name, snapshot_column
):
    """
    Reuse the snapshot-aware MERGE in the common notebook.

    Used for both Repository Metadata and Contributors.

    It uses ingested_at in Bronze and snapshot_at in Silver
    instead of relying on GitHub's updated_at field.
    """

    merge_metadata_snapshot(
        spark,
        df,
        table_name,
        snapshot_column
    )

    # Retrieve MERGE operation statistics
    metrics = spark.sql(
        f"DESCRIBE HISTORY {table_name} LIMIT 1"
    ).first()["operationMetrics"]

    inserted = int(metrics.get("numTargetRowsInserted", 0))
    updated = int(metrics.get("numTargetRowsUpdated", 0))

    return inserted, updated


# ============================================================
# 4. GENERATE LOGICAL RECORD KEYS
# ============================================================

def source_keys_from_raw(raw, repo, entity):
    """
    Generate keys belonging to the current Raw batch.

    Commits use SHA.
    All other entities use GitHub numeric IDs.

    The repository name is included to distinguish records
    belonging to different repositories.
    """

    # Identify the unique source ID
    id_col = (
        F.col("sha")
        if entity == "commits"
        else F.col("id").cast("string")
    )

    # Exclude malformed rows, missing IDs and wrong repositories
    good = (
        raw
        .filter(F.col("_corrupt_record").isNull())
        .filter(
            id_col.isNotNull()
            & (F.trim(id_col) != "")
        )
        .filter(
            F.col("_repo").isNull()
            | (F.col("_repo") == repo)
        )
    )

    # GitHub's Issues API can also return pull requests
    if entity == "issues":
        good = good.filter(
            F.col("pull_request").isNull()
        )

    # Do not pass invalid snapshot keys into Silver
    if entity in SNAPSHOT_ENTITIES:
        good = good.filter(
            F.expr(
                "try_cast(_ingested_at as timestamp)"
            ).isNotNull()
        )

    # Logical key: repository + source record ID
    return good.select(
        F.concat_ws(
            "|",
            F.lit(repo),
            id_col
        ).alias("record_key")
    ).distinct()


# ============================================================
# 5. RAW -> BRONZE PROCESSING
# ============================================================

def run_bronze_one(item):
    """
    Read one Raw JSONL file using an explicit schema.

    Steps:
    1. Read Raw data
    2. Validate required fields
    3. Quarantine invalid records
    4. Generate business keys and payload hashes
    5. Deduplicate records
    6. MERGE valid records into Bronze
    7. Log execution results
    """

    repo = item["repository"]
    entity = item["entity"]
    file_path = item["path"]

    started = now()
    execution_id = str(uuid.uuid4())
    count = 0
    t0 = time.perf_counter()

    try:
        # Read Raw JSONL with the declared PySpark schema
        # No inferSchema=True
        raw = (
            spark.read
            .schema(RAW_SCHEMAS[entity])
            .option("mode", "PERMISSIVE")
            .option(
                "columnNameOfCorruptRecord",
                "_corrupt_record"
            )
            .json(file_path)
        )

        count = raw.count()

        # Determine the logical source identifier
        raw_id = (
            F.col("sha")
            if entity == "commits"
            else F.col("id").cast("string")
        )

        # Choose the source update timestamp
        if entity == "commits":
            updated = F.col("commit.author.date")

        elif entity == "contributors":
            # Contributors have no updated_at field
            updated = F.lit(None).cast("string")

        else:
            # Issues, Pull Requests and Repository Metadata
            updated = F.col("updated_at")

        # ----------------------------------------------------
        # BRONZE VALIDATION
        # ----------------------------------------------------

        reason = (
            F.when(
                F.col("_corrupt_record").isNotNull(),
                "BAD_JSON_OR_TYPE"
            )
            .when(
                raw_id.isNull() | (F.trim(raw_id) == ""),
                "MISSING_PRIMARY_ID"
            )
            .when(
                F.col("_repo").isNotNull()
                & (F.col("_repo") != repo),
                "REPOSITORY_MISMATCH"
            )
        )

        # Exclude pull requests returned by the Issues endpoint
        if entity == "issues":
            reason = reason.when(
                F.col("pull_request").isNotNull(),
                "PULL_REQUEST_IN_ISSUES"
            )

        # Both snapshot entities require a valid collection time
        elif entity in SNAPSHOT_ENTITIES:
            reason = reason.when(
                F.expr(
                    "try_cast(_ingested_at as timestamp)"
                ).isNull(),
                "INVALID_SNAPSHOT_TIMESTAMP"
            )

        # Attach validation status and source ID
        staged = (
            raw
            .withColumn("_reason", reason)
            .withColumn("_entity_id", raw_id)
            .withColumn("_updated_str", updated)
        )

        # Select all explicitly modeled source fields
        payload_cols = [
            F.col(c)
            for c in RAW_SCHEMAS[entity].fieldNames()
            if c != "_corrupt_record"
        ]

        # ----------------------------------------------------
        # BRONZE QUARANTINE
        # ----------------------------------------------------

        failed = staged.filter(
            F.col("_reason").isNotNull()
        )

        # Preserve the rejected payload for debugging
        rejected_payload = F.coalesce(
            F.col("_corrupt_record"),
            F.to_json(F.struct(*payload_cols))
        )

        quarantined_df = failed.select(
            F.sha2(
                F.concat_ws(
                    "|",
                    F.lit(batch_id),
                    F.lit(repo),
                    F.lit(entity),
                    rejected_payload
                ),
                256
            ).alias("quarantine_id"),

            F.lit(batch_id).alias("batch_id"),
            F.lit(repo).alias("repository"),
            F.lit(entity).alias("entity"),
            F.lit(file_path).alias("source_file"),

            rejected_payload.alias("raw_payload"),
            F.col("_reason").alias("error_type"),

            F.concat(
                F.lit("Failed Bronze validation: "),
                F.col("_reason")
            ).alias("error_reason"),

            F.current_timestamp().alias("load_timestamp"),
            F.current_timestamp().alias("quarantined_at")
        )

        # Store rejected records without duplicating quarantines
        quarantined_count = quarantine(
            spark, quarantined_df, "bronze"
        )

        # ----------------------------------------------------
        # PREPARE VALID BRONZE RECORDS
        # ----------------------------------------------------

        valid = staged.filter(
            F.col("_reason").isNull()
        )

        # Hash only business fields for change detection
        business_json = F.to_json(
            F.struct(*[
                F.col(c)
                for c in BUSINESS_COLS[entity]
            ])
        )

        # Preserve the explicitly parsed API payload
        raw_json = F.to_json(
            F.struct(*payload_cols)
        )

        source = valid.select(
            # Logical primary key
            F.concat_ws(
                "|",
                F.lit(repo),
                F.col("_entity_id")
            ).alias("record_key"),

            # Source and ingestion lineage
            F.lit(repo).alias("repository"),
            F.lit(batch_id).alias("batch_id"),
            F.lit(file_path).alias("source_file"),

            # GitHub source-update timestamp
            F.expr(
                "try_cast(_updated_str as timestamp)"
            ).alias("source_updated_at"),

            # Original parsed payload and business hash
            raw_json.alias("raw_payload"),
            F.sha2(business_json, 256).alias("payload_hash"),

            # Pipeline and snapshot timestamps
            F.current_timestamp().alias("load_timestamp"),

            F.expr(
                "try_cast(_ingested_at as timestamp)"
            ).alias("ingested_at")
        )

        # ----------------------------------------------------
        # BRONZE DEDUPLICATION
        # ----------------------------------------------------

        # Snapshot datasets use collection time.
        # Core entities use their GitHub update timestamp.
        order_column = (
            "ingested_at"
            if entity in SNAPSHOT_ENTITIES
            else "source_updated_at"
        )

        ranking = Window.partitionBy(
            "record_key"
        ).orderBy(
            F.col(order_column).desc_nulls_last(),
            F.col("payload_hash").desc()
        )

        # Keep the newest version of each logical key
        source = (
            source
            .withColumn(
                "_rn",
                F.row_number().over(ranking)
            )
            .filter(F.col("_rn") == 1)
            .drop("_rn")
        )

        valid_count = source.count()

        # ----------------------------------------------------
        # BRONZE DELTA MERGE
        # ----------------------------------------------------

        if not valid_count:
            inserted, updated_count = 0, 0

        elif entity in SNAPSHOT_ENTITIES:
            # Repository Metadata and Contributors
            inserted, updated_count = merge_delta_metadata_snapshot(
                source,
                f"{CATALOG}.bronze.{entity}",
                "ingested_at"
            )

        else:
            # Commits, Issues and Pull Requests
            inserted, updated_count = merge_delta_changed(
                source,
                f"{CATALOG}.bronze.{entity}"
            )

        # ----------------------------------------------------
        # BRONZE AUDIT LOG
        # ----------------------------------------------------

        result = {
            "repository": repo,
            "entity": entity,
            "rows_processed": count,
            "valid_keys": valid_count,
            "inserted": inserted,
            "updated": updated_count,
            "upserted": inserted + updated_count,
            "quarantined": quarantined_count
        }

        log(
            spark, execution_id, batch_id,
            "raw_to_bronze", "BRONZE",
            run_date, load_type, started,
            "SUCCESS", count,
            upserted=result["upserted"],
            quarantined=quarantined_count,
            entity_type=entity,
            repository=repo,
            source_file=file_path,
            inserted=inserted,
            updated=updated_count
        )

        print(
            f"BRONZE {repo:30} {entity:13} "
            f"input={count:<6} "
            f"inserted={inserted:<5} "
            f"updated={updated_count:<5} "
            f"quarantine={quarantined_count} "
            f"({time.perf_counter()-t0:.1f}s)",
            flush=True
        )

        # Return Raw file keys so Silver can process them
        # even when Bronze MERGE makes no changes
        return result, source_keys_from_raw(
            raw, repo, entity
        )

    except Exception:
        # Record the failure before raising the error
        try:
            log(
                spark, execution_id, batch_id,
                "raw_to_bronze", "BRONZE",
                run_date, load_type, started,
                "FAILED", count,
                traceback.format_exc(),
                entity_type=entity,
                repository=repo,
                source_file=file_path
            )

        except Exception as exc:
            print(
                "BRONZE audit failure:",
                repr(exc),
                flush=True
            )

        raise


# ============================================================
# 6. BRONZE -> SILVER PROCESSING
# ============================================================

def run_silver_one(item, current_batch_keys):
    """
    Process accepted Bronze records into typed Silver tables.

    Only records belonging to the current Raw input keys
    are selected for processing.
    """

    repo = item["repository"]
    entity = item["entity"]

    started = now()
    execution_id = str(uuid.uuid4())
    count = 0
    t0 = time.perf_counter()

    try:
        # ----------------------------------------------------
        # READ THE RELEVANT BRONZE RECORDS
        # ----------------------------------------------------

        # Filtering by keys allows idempotent retries.
        # Filtering only by batch_id would miss unchanged rows.
        bronze = (
            spark.table(f"{CATALOG}.bronze.{entity}")
            .filter(F.col("repository") == repo)
            .join(
                current_batch_keys,
                "record_key",
                "left_semi"
            )
        )

        count = bronze.count()

        # Nothing to process for this repository/entity
        if count == 0:
            result = {
                "repository": repo,
                "entity": entity,
                "rows_processed": 0,
                "valid_keys": 0,
                "inserted": 0,
                "updated": 0,
                "upserted": 0,
                "quarantined": 0
            }

            log(
                spark, execution_id, batch_id,
                "bronze_to_silver", "SILVER",
                run_date, load_type, started,
                "SUCCESS", 0,
                entity_type=entity,
                repository=repo,
                source_file=item.get("path")
            )

            print(
                f"SILVER {repo:30} {entity:13} no valid keys "
                f"({time.perf_counter()-t0:.1f}s)",
                flush=True
            )

            return result

        # Parse Bronze raw_payload using the explicit Raw schema
        parsed = bronze.withColumn(
            "api",
            F.from_json(
                "raw_payload",
                RAW_SCHEMAS[entity]
            )
        )

        # Helper for converting ISO timestamp strings
        ts = lambda expr: F.expr(
            f"try_cast({expr} as timestamp)"
        )

        # ----------------------------------------------------
        # SILVER: COMMITS
        # ----------------------------------------------------

        if entity == "commits":

            silver = parsed.select(
                "record_key",
                "repository",
                F.col("api.sha").alias("commit_sha"),
                F.col("api.author.id").alias("author_user_id"),
                ts("api.commit.author.date").alias("commit_timestamp"),
                F.col("api.commit.message").alias("message"),
                "batch_id",
                "source_updated_at",
                "payload_hash",
                F.current_timestamp().alias("load_timestamp")
            )

            reason = (
                F.when(
                    F.col("commit_sha").isNull(),
                    "NULL_COMMIT_SHA"
                )
                .when(
                    F.col("commit_timestamp").isNull(),
                    "INVALID_COMMIT_TIMESTAMP"
                )
            )

        # ----------------------------------------------------
        # SILVER: ISSUES
        # ----------------------------------------------------

        elif entity == "issues":

            silver = parsed.select(
                "record_key",
                "repository",
                F.col("api.id").alias("issue_id"),
                F.col("api.number").alias("issue_number"),
                F.col("api.user.id").alias("author_user_id"),
                F.col("api.title").alias("title"),
                F.col("api.state").alias("state"),
                ts("api.created_at").alias("created_at"),
                ts("api.updated_at").alias("updated_at"),
                ts("api.closed_at").alias("closed_at"),
                "batch_id",
                "source_updated_at",
                "payload_hash",
                F.current_timestamp().alias("load_timestamp")
            )

            reason = (
                F.when(
                    F.col("issue_id").isNull(),
                    "NULL_ISSUE_ID"
                )
                .when(
                    F.col("created_at").isNull()
                    | F.col("updated_at").isNull(),
                    "INVALID_TIMESTAMP"
                )
            )

        # ----------------------------------------------------
        # SILVER: REPOSITORY METADATA
        # ----------------------------------------------------

        elif entity == "repo_metadata":

            # Keep repository metrics for analytics
            silver = parsed.select(
                "record_key",
                "repository",

                F.col("api.id").alias("repo_id"),
                F.col("api.description").alias("description"),
                F.col("api.html_url").alias("html_url"),
                F.col("api.language").alias("language"),
                F.col("api.default_branch").alias("default_branch"),

                F.col("api.stargazers_count").alias("stars_count"),
                F.col("api.forks_count").alias("forks_count"),
                F.col("api.open_issues_count").alias("open_issues_count"),
                F.col("api.subscribers_count").alias("subscribers_count"),
                F.col("api.archived").alias("archived"),

                ts("api.created_at").alias("created_at"),
                ts("api.updated_at").alias("updated_at"),
                ts("api.pushed_at").alias("pushed_at"),

                F.col("ingested_at").alias("snapshot_at"),

                "batch_id",
                "source_updated_at",
                "payload_hash",
                F.current_timestamp().alias("load_timestamp")
            )

            # FIXED: Metadata validation is inside
            # the Repository Metadata branch, not Contributors.
            reason = (
                F.when(
                    F.col("repo_id").isNull(),
                    "NULL_REPOSITORY_ID"
                )
                .when(
                    F.col("created_at").isNull(),
                    "INVALID_CREATED_AT"
                )
                .when(
                    F.col("snapshot_at").isNull(),
                    "INVALID_SNAPSHOT_TIMESTAMP"
                )
            )

        # ----------------------------------------------------
        # SILVER: CONTRIBUTORS
        # ----------------------------------------------------

        elif entity == "contributors":

            # Keep numeric IDs, not usernames/profile URLs.
            silver = parsed.select(
                "record_key",
                "repository",

                F.col("api.id").alias("contributor_id"),
                F.col("api.contributions").alias("contributions"),
                F.col("api.type").alias("account_type"),

                # Snapshot collection timestamp from Bronze
                F.col("ingested_at").alias("snapshot_at"),

                "batch_id",
                "payload_hash",
                F.current_timestamp().alias("load_timestamp")
            )

            # Contributors have their own validation rules
            reason = (
                F.when(
                    F.col("contributor_id").isNull(),
                    "NULL_CONTRIBUTOR_ID"
                )
                .when(
                    F.col("contributions").isNull()
                    | (F.col("contributions") < 0),
                    "INVALID_CONTRIBUTIONS"
                )
                .when(
                    F.col("snapshot_at").isNull(),
                    "INVALID_SNAPSHOT_TIMESTAMP"
                )
            )

        # ----------------------------------------------------
        # SILVER: PULL REQUESTS
        # ----------------------------------------------------

        elif entity == "pulls":

            silver = parsed.select(
                "record_key",
                "repository",

                F.col("api.id").alias("pr_id"),
                F.col("api.number").alias("pr_number"),
                F.col("api.user.id").alias("author_user_id"),
                F.col("api.title").alias("title"),
                F.col("api.state").alias("state"),
                F.col("api.draft").alias("draft"),

                ts("api.created_at").alias("created_at"),
                ts("api.updated_at").alias("updated_at"),
                ts("api.closed_at").alias("closed_at"),
                ts("api.merged_at").alias("merged_at"),

                "batch_id",
                "source_updated_at",
                "payload_hash",
                F.current_timestamp().alias("load_timestamp")
            )

            reason = (
                F.when(
                    F.col("pr_id").isNull(),
                    "NULL_PR_ID"
                )
                .when(
                    F.col("created_at").isNull()
                    | F.col("updated_at").isNull(),
                    "INVALID_TIMESTAMP"
                )
            )

        else:
            # Reject unsupported entities
            raise ValueError(
                f"Unknown entity for Silver: {entity}"
            )

        # ----------------------------------------------------
        # SILVER QUARANTINE
        # ----------------------------------------------------

        # Mark records failing Silver validation
        staged = silver.withColumn("_reason", reason)

        failed = staged.filter(
            F.col("_reason").isNotNull()
        )

        # Join the original Bronze payload for debugging
        q = (
            failed
            .join(
                bronze.select("record_key", "raw_payload"),
                "record_key",
                "left"
            )
            .select(
                F.sha2(
                    F.concat_ws(
                        "|",
                        F.lit(batch_id),
                        F.lit(entity),
                        "record_key",
                        "_reason"
                    ),
                    256
                ).alias("quarantine_id"),

                F.lit(batch_id).alias("batch_id"),
                F.col("repository"),
                F.lit(entity).alias("entity"),
                F.col("record_key"),
                F.col("raw_payload"),
                F.col("_reason").alias("error_type"),

                F.concat(
                    F.lit("Silver validation failure: "),
                    F.col("_reason")
                ).alias("error_reason"),

                F.current_timestamp().alias("load_timestamp"),
                F.current_timestamp().alias("quarantined_at")
            )
        )

        quarantined_count = quarantine(
            spark, q, "silver"
        )

        # Continue only with valid records
        valid = (
            staged
            .filter(F.col("_reason").isNull())
            .drop("_reason")
        )

        # ----------------------------------------------------
        # SILVER DEDUPLICATION
        # ----------------------------------------------------

        # Contributors and metadata use snapshot timestamps.
        # Other entities use source_updated_at.
        order_column = (
            "snapshot_at"
            if entity in SNAPSHOT_ENTITIES
            else "source_updated_at"
        )

        ranking = Window.partitionBy(
            "record_key"
        ).orderBy(
            F.col(order_column).desc_nulls_last(),
            F.col("payload_hash").desc()
        )

        valid = (
            valid
            .withColumn(
                "_rn",
                F.row_number().over(ranking)
            )
            .filter(F.col("_rn") == 1)
            .drop("_rn")
        )

        valid_count = valid.count()

        # ----------------------------------------------------
        # SILVER DELTA MERGE
        # ----------------------------------------------------

        if not valid_count:
            inserted, updated_count = 0, 0

        elif entity in SNAPSHOT_ENTITIES:
            # Snapshot-aware MERGE for Contributors and Metadata
            inserted, updated_count = merge_delta_metadata_snapshot(
                valid,
                f"{CATALOG}.silver.{entity}",
                "snapshot_at"
            )

        else:
            # Timestamp-based MERGE for core entities
            inserted, updated_count = merge_delta_changed(
                valid,
                f"{CATALOG}.silver.{entity}"
            )

        # ----------------------------------------------------
        # SILVER AUDIT LOG
        # ----------------------------------------------------

        result = {
            "repository": repo,
            "entity": entity,
            "rows_processed": count,
            "valid_keys": valid_count,
            "inserted": inserted,
            "updated": updated_count,
            "upserted": inserted + updated_count,
            "quarantined": quarantined_count
        }

        log(
            spark, execution_id, batch_id,
            "bronze_to_silver", "SILVER",
            run_date, load_type, started,
            "SUCCESS", count,
            upserted=result["upserted"],
            quarantined=quarantined_count,
            entity_type=entity,
            repository=repo,
            source_file=item.get("path"),
            inserted=inserted,
            updated=updated_count
        )

        print(
            f"SILVER {repo:30} {entity:13} "
            f"input={count:<6} "
            f"inserted={inserted:<5} "
            f"updated={updated_count:<5} "
            f"quarantine={quarantined_count} "
            f"({time.perf_counter()-t0:.1f}s)",
            flush=True
        )

        return result

    except Exception:
        # Record the error in the execution log
        try:
            log(
                spark, execution_id, batch_id,
                "bronze_to_silver", "SILVER",
                run_date, load_type, started,
                "FAILED", count,
                traceback.format_exc(),
                entity_type=entity,
                repository=repo,
                source_file=item.get("path")
            )

        except Exception as exc:
            print(
                "SILVER audit failure:",
                repr(exc),
                flush=True
            )

        raise


In [0]:
# Cell 6: EXECUTE. All Bronze first, then all Silver. One Spark session, no child runs.
# Click Run All once with the widgets set to your desired batch.

overall_start = now()
overall_execution_id = str(uuid.uuid4())
bronze_results, silver_results = [], []
keys_by_pair = {}
processed = 0
try:
    ensure_entity_tables(spark)

    # Select only the files requested by run_scope
    # METADATA_ONLY excludes the existing 9 core files

    # Select which GitHub datasets to process
    # This prevents unrelated datasets from being processed

    if run_scope == "CORE_ONLY":
        # Commits, Issues, and Pull Requests
        selected_files = files

    elif run_scope == "METADATA_ONLY":
        # Only Repository Metadata
        selected_files = metadata_files

    elif run_scope == "CONTRIBUTORS_ONLY":
        # Only Contributors (3 files, 1,151 records)
        selected_files = contributors_files

    elif run_scope == "ALL":
        # Process all five entities
        selected_files = files + metadata_files + contributors_files

    else:
        # Stop if the scope is invalid
        raise ValueError("Invalid run_scope")

    # Safety check: Contributors-only must never process other entities
    if run_scope == "CONTRIBUTORS_ONLY":
        assert len(selected_files) == 3
        assert all(
            item["entity"] == "contributors"
            for item in selected_files
        )


    print(
        f"\n=========== BRONZE — {len(selected_files)} FILES ===========",
        flush=True
    )

    for item in selected_files:

        result, keys = run_bronze_one(item)
        bronze_results.append(result)
        keys_by_pair[(item["repository"], item["entity"])] = keys
        processed += result["rows_processed"]


    # Process the same selected files through Silver
    print(
        f"\n=========== SILVER — {len(selected_files)} FILES ===========",
        flush=True
    )

    for item in selected_files:

        pair = (item["repository"], item["entity"])
        result = run_silver_one(item, keys_by_pair[pair])
        silver_results.append(result)
        processed += result["rows_processed"]

    log(spark, overall_execution_id, batch_id, "driver_single_session",
        "PIPELINE", run_date, load_type, overall_start, "SUCCESS", processed,
        upserted=sum(x["upserted"] for x in bronze_results + silver_results),
        quarantined=sum(x["quarantined"] for x in bronze_results + silver_results))
except Exception:
    error = traceback.format_exc()
    print("\nPIPELINE FAILED:\n" + error, flush=True)
    try:
        log(spark, overall_execution_id, batch_id, "driver_single_session",
            "PIPELINE", run_date, load_type, overall_start, "FAILED", processed,
            error=error,
            upserted=sum(x["upserted"] for x in bronze_results + silver_results),
            quarantined=sum(x["quarantined"] for x in bronze_results + silver_results))
    except Exception as audit_error:
        print("PIPELINE AUDIT FAILED:", repr(audit_error), flush=True)
    raise

summary = {
    "status": "SUCCESS",
    "batch_id": batch_id,
    "load_type": load_type,
    "bronze_processed": sum(x["rows_processed"] for x in bronze_results),
    "bronze_upserted": sum(x["upserted"] for x in bronze_results),
    "silver_processed": sum(x["rows_processed"] for x in silver_results),
    "silver_upserted": sum(x["upserted"] for x in silver_results),
    "bronze_quarantined": sum(x["quarantined"] for x in bronze_results),
    "silver_quarantined": sum(x["quarantined"] for x in silver_results),
}
print("\n=========== COMPLETE ===========", flush=True)
print(json.dumps(summary, indent=2), flush=True)
# Do not call dbutils.notebook.exit here: leaving output visible for manual Run All.


=========== BRONZE — 15 FILES ===========
BRONZE apache/spark                   commits       input=4775   inserted=0     updated=0     quarantine=0 (10.1s)
BRONZE apache/spark                   issues        input=110    inserted=0     updated=0     quarantine=0 (8.5s)
BRONZE apache/spark                   pulls         input=8403   inserted=0     updated=0     quarantine=0 (10.1s)
BRONZE pandas-dev/pandas              commits       input=2777   inserted=0     updated=0     quarantine=0 (8.6s)
BRONZE pandas-dev/pandas              issues        input=3135   inserted=0     updated=0     quarantine=0 (8.8s)
BRONZE pandas-dev/pandas              pulls         input=4739   inserted=0     updated=0     quarantine=0 (10.5s)
BRONZE scikit-learn/scikit-learn      commits       input=1054   inserted=0     updated=0     quarantine=0 (10.6s)
BRONZE scikit-learn/scikit-learn      issues        input=1106   inserted=0     updated=0     quarantine=0 (8.5s)
BRONZE scikit-learn/scikit-learn      pul

In [0]:

# =====================================================
# PHASE 2: EXECUTION LOG VERIFICATION
# Read-only: does not modify any existing data
# =====================================================

from pyspark.sql import functions as F

table_name = "github_health.ops.pipeline_execution_logs"
logs = spark.table(table_name)

# 1. Show the actual logging table structure
print("========== CURRENT LOG SCHEMA ==========")
logs.printSchema()

# 2. Check the required columns
required_columns = [
    "entity_type",
    "repository",
    "source_file",
    "inserted_rows",
    "updated_rows"
]

print("\n========== COLUMN VERIFICATION ==========")

for column in required_columns:
    if column in logs.columns:
        print(f"PASS: {column}")
    else:
        print(f"MISSING: {column}")

# 3. Display the latest incremental execution logs
print("\n========== INCREMENTAL EXECUTION LOGS ==========")

# Include only fields that actually exist
columns_to_show = [
    "batch_id",
    "layer",
    "entity_type",
    "repository",
    "source_file",
    "status",
    "rows_processed",
    "inserted_rows",
    "updated_rows",
    "upserted_rows",
    "start_time",
    "end_time"
]

available = [
    column for column in columns_to_show
    if column in logs.columns
]

recent_logs = (
    logs
    .filter(F.col("batch_id") == "incremental_20261008_001")
    .orderBy(F.col("start_time").desc())
    .select(*available)
    .limit(20)
)

display(recent_logs)


========== CURRENT LOG SCHEMA ==========
root
 |-- execution_id: string (nullable = false)
 |-- batch_id: string (nullable = false)
 |-- notebook_name: string (nullable = true)
 |-- layer: string (nullable = true)
 |-- run_date: date (nullable = true)
 |-- load_type: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- rows_processed: long (nullable = true)
 |-- error_message: string (nullable = true)
 |-- upserted_rows: long (nullable = true)
 |-- quarantined_rows: long (nullable = true)
 |-- entity_type: string (nullable = true)
 |-- repository: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- inserted_rows: long (nullable = true)
 |-- updated_rows: long (nullable = true)


========== COLUMN VERIFICATION ==========
PASS: entity_type
PASS: repository
PASS: source_file
PASS: inserted_rows
PASS: updated_rows

========== INCREMENTAL EXECUTION LOGS ==========


batch_id,layer,entity_type,repository,source_file,status,rows_processed,inserted_rows,updated_rows,upserted_rows,start_time,end_time
incremental_20261008_001,SILVER,pulls,scikit-learn/scikit-learn,/Volumes/github_health/raw/bronze_data/bronze/pulls/scikit-learn__scikit-learn/run_date=2026-10-08/pulls_incremental_incremental_20261008_001.jsonl,SUCCESS,9,0,0,0,2026-10-10T21:29:34.896Z,2026-10-10T21:29:44.331Z
incremental_20261008_001,SILVER,issues,scikit-learn/scikit-learn,/Volumes/github_health/raw/bronze_data/bronze/issues/scikit-learn__scikit-learn/run_date=2026-10-08/issues_incremental_incremental_20261008_001.jsonl,SUCCESS,2,0,0,0,2026-10-10T21:29:24.414Z,2026-10-10T21:29:33.707Z
incremental_20261008_001,SILVER,commits,scikit-learn/scikit-learn,/Volumes/github_health/raw/bronze_data/bronze/commits/scikit-learn__scikit-learn/run_date=2026-10-08/commits_incremental_incremental_20261008_001.jsonl,SUCCESS,0,0,0,0,2026-10-10T21:29:22.047Z,2026-10-10T21:29:23.051Z
incremental_20261008_001,SILVER,pulls,pandas-dev/pandas,/Volumes/github_health/raw/bronze_data/bronze/pulls/pandas-dev__pandas/run_date=2026-10-08/pulls_incremental_incremental_20261008_001.jsonl,SUCCESS,14,0,0,0,2026-10-10T21:29:10.874Z,2026-10-10T21:29:20.820Z
incremental_20261008_001,SILVER,issues,pandas-dev/pandas,/Volumes/github_health/raw/bronze_data/bronze/issues/pandas-dev__pandas/run_date=2026-10-08/issues_incremental_incremental_20261008_001.jsonl,SUCCESS,3,0,0,0,2026-10-10T21:28:58.548Z,2026-10-10T21:29:09.580Z
incremental_20261008_001,SILVER,commits,pandas-dev/pandas,/Volumes/github_health/raw/bronze_data/bronze/commits/pandas-dev__pandas/run_date=2026-10-08/commits_incremental_incremental_20261008_001.jsonl,SUCCESS,2,0,0,0,2026-10-10T21:28:47.544Z,2026-10-10T21:28:57.136Z
incremental_20261008_001,SILVER,pulls,apache/spark,/Volumes/github_health/raw/bronze_data/bronze/pulls/apache__spark/run_date=2026-10-08/pulls_incremental_incremental_20261008_001.jsonl,SUCCESS,28,0,0,0,2026-10-10T21:28:36.283Z,2026-10-10T21:28:46.313Z
incremental_20261008_001,SILVER,issues,apache/spark,/Volumes/github_health/raw/bronze_data/bronze/issues/apache__spark/run_date=2026-10-08/issues_incremental_incremental_20261008_001.jsonl,SUCCESS,0,0,0,0,2026-10-10T21:28:33.951Z,2026-10-10T21:28:34.921Z
incremental_20261008_001,SILVER,commits,apache/spark,/Volumes/github_health/raw/bronze_data/bronze/commits/apache__spark/run_date=2026-10-08/commits_incremental_incremental_20261008_001.jsonl,SUCCESS,2,0,0,0,2026-10-10T21:28:22.372Z,2026-10-10T21:28:32.660Z
incremental_20261008_001,BRONZE,pulls,scikit-learn/scikit-learn,/Volumes/github_health/raw/bronze_data/bronze/pulls/scikit-learn__scikit-learn/run_date=2026-10-08/pulls_incremental_incremental_20261008_001.jsonl,SUCCESS,9,0,0,0,2026-10-10T21:28:12.657Z,2026-10-10T21:28:21.056Z
